# Classical Approaches to Exploration

Before reaching for reinforcement learning, it's worth understanding the classical
alternatives. BFS, A*, and frontier-based exploration are not just teaching examples:
they remain competitive baselines in structured environments and are often the right
tool when the map is fully or partially known.

In [ ]:
import heapq
import random
from collections import deque

import matplotlib.pyplot as plt
import numpy as np

## The Gridworld

We'll work in a 2D grid throughout this notebook.
Each cell is either free (0) or a wall (1).
An agent starts at a fixed cell and moves in four directions: up, down, left, right.

In [ ]:
class GridWorld:
    def __init__(self, grid, start, goal):
        """
        grid: 2D numpy array, 0=free, 1=wall
        start: (row, col) tuple
        goal: (row, col) tuple
        """
        self.grid = np.array(grid)
        self.rows, self.cols = self.grid.shape
        self.start = start
        self.goal = goal

    def neighbors(self, pos):
        r, c = pos
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.rows and 0 <= nc < self.cols and self.grid[nr, nc] == 0:
                yield (nr, nc)

    def visualize(self, path=None, visited=None, title=""):
        display = np.zeros((*self.grid.shape, 3), dtype=np.uint8)
        display[self.grid == 0] = [255, 255, 255]  # free: white
        display[self.grid == 1] = [50, 50, 50]     # wall: dark gray

        if visited:
            for r, c in visited:
                display[r, c] = [180, 220, 255]  # visited: light blue
        if path:
            for r, c in path:
                display[r, c] = [80, 200, 80]    # path: green

        sr, sc = self.start
        gr, gc = self.goal
        display[sr, sc] = [255, 165, 0]   # start: orange
        display[gr, gc] = [220, 50, 50]   # goal: red

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(display, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
        plt.tight_layout()
        plt.show()


# Build a small maze
GRID = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 1, 0, 0, 0, 1, 0],
    [0, 1, 0, 0, 1, 1, 1, 0, 0, 0],
    [0, 1, 1, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 1, 0, 1, 1, 0],
    [0, 1, 1, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
]

world = GridWorld(GRID, start=(0, 0), goal=(9, 9))
world.visualize(title="Gridworld (orange=start, red=goal)")

## Breadth-First Search

BFS explores the grid layer by layer, expanding all cells at distance 1 before
any at distance 2. This guarantees the shortest path in an unweighted grid.

The queue is a FIFO (first-in, first-out) structure: cells are explored in the
order they were discovered.

In [ ]:
def bfs(world):
    queue = deque([(world.start, [world.start])])
    visited = {world.start}

    while queue:
        pos, path = queue.popleft()
        if pos == world.goal:
            return path, visited
        for nb in world.neighbors(pos):
            if nb not in visited:
                visited.add(nb)
                queue.append((nb, path + [nb]))

    return None, visited  # no path found


bfs_path, bfs_visited = bfs(world)
print(f"BFS: path length = {len(bfs_path)}, cells explored = {len(bfs_visited)}")
world.visualize(path=bfs_path, visited=bfs_visited, title="BFS (blue=explored, green=path)")

## Depth-First Search

DFS uses a stack (LIFO) instead of a queue. It dives deep along one branch before
backtracking. DFS often finds *a* path quickly but rarely the shortest one.
Compare the path length and exploration count to BFS.

In [ ]:
def dfs(world):
    stack = [(world.start, [world.start])]
    visited = set()

    while stack:
        pos, path = stack.pop()
        if pos in visited:
            continue
        visited.add(pos)
        if pos == world.goal:
            return path, visited
        for nb in world.neighbors(pos):
            if nb not in visited:
                stack.append((nb, path + [nb]))

    return None, visited


dfs_path, dfs_visited = dfs(world)
print(f"DFS: path length = {len(dfs_path)}, cells explored = {len(dfs_visited)}")
print(f"BFS: path length = {len(bfs_path)}, cells explored = {len(bfs_visited)}")
world.visualize(path=dfs_path, visited=dfs_visited, title="DFS (blue=explored, green=path)")

## A* Search

A* combines BFS's optimality guarantee with a heuristic that guides search toward the goal.
Each cell is scored as `f(n) = g(n) + h(n)` where:
- `g(n)` = cost from start to this cell (number of steps)
- `h(n)` = estimated cost from this cell to goal (the heuristic)

With Manhattan distance as the heuristic, A* expands far fewer cells than BFS
while still finding the shortest path.

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(world):
    start, goal = world.start, world.goal
    # heap entries: (f_score, g_score, position, path)
    heap = [(manhattan(start, goal), 0, start, [start])]
    visited = {}

    while heap:
        f, g, pos, path = heapq.heappop(heap)
        if pos in visited:
            continue
        visited[pos] = g
        if pos == goal:
            return path, set(visited.keys())
        for nb in world.neighbors(pos):
            if nb not in visited:
                ng = g + 1
                heapq.heappush(heap, (ng + manhattan(nb, goal), ng, nb, path + [nb]))

    return None, set(visited.keys())


astar_path, astar_visited = astar(world)
print(f"A*:  path length = {len(astar_path)}, cells explored = {len(astar_visited)}")
print(f"BFS: path length = {len(bfs_path)}, cells explored = {len(bfs_visited)}")
world.visualize(path=astar_path, visited=astar_visited, title="A* (blue=explored, green=path)")

## Frontier-Based Exploration

The approaches above assume you know the full map. In real robotics and autonomous
exploration, the map is unknown. You only know what the agent has already seen.

Frontier-based exploration works as follows:
1. Start with all cells marked unknown.
2. As the agent moves, mark nearby cells as known (free or wall).
3. A **frontier** is a free known cell adjacent to an unknown cell.
4. Pick the nearest frontier and navigate there using A*.
5. Repeat until no frontiers remain.

This is a greedy strategy: it maximizes the information gained per step.

In [ ]:
def frontier_exploration(world, vision_radius=2):
    """
    Simulates frontier-based exploration on world.grid.
    vision_radius: cells revealed around the agent at each step.
    Returns coverage history (list of % cells known after each move).
    """
    rows, cols = world.grid.shape
    total_free = np.sum(world.grid == 0)

    known = np.full((rows, cols), -1, dtype=int)  # -1=unknown, 0=free, 1=wall
    agent = world.start
    coverage_history = []

    def reveal(pos):
        r, c = pos
        for dr in range(-vision_radius, vision_radius + 1):
            for dc in range(-vision_radius, vision_radius + 1):
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols:
                    known[nr, nc] = world.grid[nr, nc]

    def get_frontiers():
        frontiers = []
        for r in range(rows):
            for c in range(cols):
                if known[r, c] == 0:  # known free cell
                    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < rows and 0 <= nc < cols and known[nr, nc] == -1:
                            frontiers.append((r, c))
                            break
        return frontiers

    def navigate_to(target):
        """A* using only known cells."""
        heap = [(manhattan(agent, target), 0, agent, [agent])]
        visited_nav = set()
        while heap:
            _, g, pos, path = heapq.heappop(heap)
            if pos in visited_nav:
                continue
            visited_nav.add(pos)
            if pos == target:
                return path
            r, c = pos
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r + dr, c + dc
                if (0 <= nr < rows and 0 <= nc < cols
                        and known[nr, nc] == 0 and (nr, nc) not in visited_nav):
                    heapq.heappush(heap, (
                        g + 1 + manhattan((nr,nc), target), g + 1, (nr, nc), path + [(nr,nc)]
                    ))
        return None

    reveal(agent)
    steps = 0
    max_steps = 2000

    while steps < max_steps:
        known_free = np.sum(known == 0)
        coverage_history.append(known_free / total_free * 100)

        frontiers = get_frontiers()
        if not frontiers:
            break

        # Pick the nearest frontier
        target = min(frontiers, key=lambda f: manhattan(agent, f))
        path = navigate_to(target)
        if path is None or len(path) <= 1:
            frontiers.remove(target)
            if not frontiers:
                break
            continue

        # Follow the path
        for pos in path[1:]:
            agent = pos
            reveal(agent)
            steps += 1

    coverage_history.append(np.sum(known == 0) / total_free * 100)
    return coverage_history


coverage = frontier_exploration(world, vision_radius=2)
print(f"Final coverage: {coverage[-1]:.1f}% of free cells")

## Hands-On: Coverage vs. Steps on a Larger Grid

In [ ]:
def random_grid(rows, cols, wall_prob=0.25, seed=42):
    rng = np.random.default_rng(seed)
    grid = (rng.random((rows, cols)) < wall_prob).astype(int)
    grid[0, 0] = 0   # ensure start is free
    grid[-1, -1] = 0 # ensure goal is free
    return grid


big_grid = random_grid(20, 20, wall_prob=0.2)
big_world = GridWorld(big_grid, start=(0, 0), goal=(19, 19))

# Compare different vision radii
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for radius, ax in zip([1, 3], axes):
    cov = frontier_exploration(big_world, vision_radius=radius)
    ax.plot(range(len(cov)), cov)
    ax.set_xlabel("Steps")
    ax.set_ylabel("Coverage (%)")
    ax.set_title(f"Vision radius = {radius}")
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)
    print(f"radius={radius}: final coverage {cov[-1]:.1f}% in {len(cov)} steps")

plt.tight_layout()
plt.show()

## Occupancy Maps

A binary known/unknown map is fine for simulation, but real sensors are noisy.
An **occupancy map** stores a probability per cell: the estimated probability
that the cell is occupied.

Cells start at 0.5 (maximum uncertainty). After observing a cell:
- If the sensor says free: probability decreases toward 0
- If the sensor says occupied: probability increases toward 1

Multiple observations update the same cell using a log-odds update rule,
which keeps probabilities from saturating at exactly 0 or 1.

In [ ]:
def log_odds(p):
    return np.log(p / (1 - p + 1e-10) + 1e-10)


def prob_from_log_odds(lo):
    return np.exp(lo) / (1 + np.exp(lo))


rows, cols = 10, 10
occupancy_log_odds = np.zeros((rows, cols))  # 0 = log_odds(0.5) = uninformative prior

# Simulate a noisy sensor: true map + 10% false positive/negative rate
true_grid = np.array(GRID)
noise_rate = 0.10

for _ in range(5):  # 5 passes of the whole map
    for r in range(rows):
        for c in range(cols):
            true_occupied = true_grid[r, c] == 1
            sensor_occupied = true_occupied != (np.random.rand() < noise_rate)
            if sensor_occupied:
                occupancy_log_odds[r, c] += log_odds(0.9)  # occupied observation
            else:
                occupancy_log_odds[r, c] += log_odds(0.1)  # free observation

occupancy_probs = prob_from_log_odds(occupancy_log_odds)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(true_grid, cmap="gray_r", interpolation="nearest")
axes[0].set_title("True Map")
axes[0].axis("off")
axes[1].imshow(occupancy_probs, cmap="RdYlGn_r", vmin=0, vmax=1, interpolation="nearest")
axes[1].set_title("Occupancy Map (after 5 noisy passes)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("Cells with p > 0.8 (likely wall):", np.sum(occupancy_probs > 0.8))
print("True walls:", np.sum(true_grid == 1))

## GridWorld with Full Reset and Step Interface

The sections above used a lightweight helper. Here we build a richer `GridWorld` class
with `reset()`, `step(action)`, and `render()` interfaces that match conventions used
in gymnasium environments. This makes it easy to swap in later.

In [ ]:
import heapq
import random
from collections import deque

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np


class GridWorld:
    """
    A 2-D grid environment with reset / step / render.

    Parameters
    ----------
    width, height : int
        Dimensions of the grid (columns, rows).
    obstacles : list of (row, col) tuples
        Cells that are walls and cannot be entered.
    start : (row, col)   -- default (0, 0)
    goal  : (row, col)   -- default (height-1, width-1)
    """

    ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right
    ACTION_NAMES = ["up", "down", "left", "right"]

    def __init__(self, width, height, obstacles, start=None, goal=None):
        self.width = width
        self.height = height
        self.obstacles = set(map(tuple, obstacles))
        self.start = tuple(start) if start else (0, 0)
        self.goal = tuple(goal) if goal else (height - 1, width - 1)

        # Build numpy grid: 0=free, 1=wall
        self.grid = np.zeros((height, width), dtype=int)
        for r, c in self.obstacles:
            self.grid[r, c] = 1

        self._agent_pos = None

    # ------------------------------------------------------------------ #
    def reset(self):
        """Place the agent at the start position and return the state."""
        self._agent_pos = self.start
        return self._agent_pos

    def step(self, action):
        """
        Move the agent.

        Parameters
        ----------
        action : int  0=up 1=down 2=left 3=right

        Returns
        -------
        next_state, reward, done
        """
        if self._agent_pos is None:
            raise RuntimeError("Call reset() before step().")

        dr, dc = self.ACTIONS[action]
        r, c = self._agent_pos
        nr, nc = r + dr, c + dc

        # Stay in place if out of bounds or wall
        if 0 <= nr < self.height and 0 <= nc < self.width and self.grid[nr, nc] == 0:
            self._agent_pos = (nr, nc)

        done = (self._agent_pos == self.goal)
        reward = 1.0 if done else -0.01
        return self._agent_pos, reward, done

    def neighbors(self, pos):
        """Yield free neighbors of pos (used by search algorithms)."""
        r, c = pos
        for dr, dc in self.ACTIONS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.height and 0 <= nc < self.width and self.grid[nr, nc] == 0:
                yield (nr, nc)

    def render(self, path=None, visited=None, title="", figsize=(6, 6)):
        """
        Render the grid with matplotlib.

        path     : list of (r, c) tuples -- shown in green
        visited  : set  of (r, c) tuples -- shown in light blue
        """
        display = np.zeros((self.height, self.width, 3), dtype=np.uint8)
        display[self.grid == 0] = [255, 255, 255]   # free: white
        display[self.grid == 1] = [50,  50,  50]    # wall: dark gray

        if visited:
            for r, c in visited:
                display[r, c] = [180, 220, 255]     # visited: light blue
        if path:
            for r, c in path:
                display[r, c] = [80,  200,  80]     # path: green

        sr, sc = self.start
        gr, gc = self.goal
        display[sr, sc] = [255, 165,   0]           # start: orange
        display[gr, gc] = [220,  50,  50]           # goal:  red

        if self._agent_pos and self._agent_pos not in (self.start, self.goal):
            ar, ac = self._agent_pos
            display[ar, ac] = [120,  80, 200]       # agent: purple

        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(display, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")

        patches = [
            mpatches.Patch(color=[1, .65, 0], label="Start"),
            mpatches.Patch(color=[.86, .2, .2], label="Goal"),
            mpatches.Patch(color=[.7, .86, 1], label="Visited"),
            mpatches.Patch(color=[.31, .78, .31], label="Path"),
        ]
        ax.legend(handles=patches, loc="upper right", fontsize=7,
                  framealpha=0.8, handlelength=1)
        plt.tight_layout()
        plt.show()


# ------------------------------------------------------------------ #
# Build a 12x12 demo world
obstacles_demo = [
    (1,1),(1,2),(1,4),(1,5),(1,6),(1,8),
    (2,4),(2,8),
    (3,1),(3,3),(3,4),(3,8),
    (4,1),(4,3),(4,5),(4,6),(4,7),
    (5,1),(5,2),(5,5),(5,9),
    (6,3),(6,4),(6,7),(6,8),
    (7,1),(7,2),(7,4),(7,6),
    (8,1),(8,4),(8,7),(8,8),(8,9),
    (9,3),(9,7),
    (10,1),(10,2),(10,5),(10,6),(10,8),
    (11,3),
]

gw = GridWorld(width=12, height=12, obstacles=obstacles_demo,
               start=(0, 0), goal=(11, 11))
gw.reset()
gw.render(title="GridWorld (orange=start, red=goal, dark=wall)")

## BFS on the New GridWorld

`bfs(grid, start, goal)` takes a `GridWorld` instance, explores layer by layer,
and returns the **shortest path** plus the set of all visited cells.

In [ ]:
def bfs(grid, start, goal):
    """
    Breadth-first search on a GridWorld.

    Returns
    -------
    path    : list of (r, c) from start to goal (inclusive), or None
    visited : set of all cells examined
    """
    queue = deque([(start, [start])])
    visited = {start}

    while queue:
        pos, path = queue.popleft()
        if pos == goal:
            return path, visited
        for nb in grid.neighbors(pos):
            if nb not in visited:
                visited.add(nb)
                queue.append((nb, path + [nb]))

    return None, visited


bfs_path, bfs_visited = bfs(gw, gw.start, gw.goal)
print(f"BFS: path length = {len(bfs_path)}, cells explored = {len(bfs_visited)}")
gw.render(path=bfs_path, visited=bfs_visited,
          title=f"BFS shortest path (length {len(bfs_path)})")

## DFS: A Path, but Not the Shortest

DFS uses a LIFO stack. It races ahead on one branch before backtracking.
It finds *a* solution faster in lucky cases, but offers no length guarantee.
Compare path length and visit count to BFS above.

In [ ]:
def dfs(grid, start, goal):
    """
    Depth-first search on a GridWorld.

    Returns
    -------
    path    : list of (r, c) from start to goal (inclusive), or None
    visited : set of all cells examined
    """
    stack = [(start, [start])]
    visited = set()

    while stack:
        pos, path = stack.pop()
        if pos in visited:
            continue
        visited.add(pos)
        if pos == goal:
            return path, visited
        for nb in grid.neighbors(pos):
            if nb not in visited:
                stack.append((nb, path + [nb]))

    return None, visited


dfs_path, dfs_visited = dfs(gw, gw.start, gw.goal)
print(f"DFS: path length = {len(dfs_path)},  cells explored = {len(dfs_visited)}")
print(f"BFS: path length = {len(bfs_path)},  cells explored = {len(bfs_visited)}")
print(f"DFS overhead: {len(dfs_path) - len(bfs_path)} extra steps vs optimal")
gw.render(path=dfs_path, visited=dfs_visited,
          title=f"DFS path (length {len(dfs_path)}) vs BFS optimal ({len(bfs_path)})")

## A* with Manhattan Heuristic

A* scores each candidate cell as `f = g + h` where `g` is the cost-so-far and
`h` is an admissible heuristic (never overestimates true cost). With Manhattan
distance on a 4-connected grid the heuristic is admissible, so A* is both
complete and optimal -- and typically expands far fewer nodes than BFS.

The visualization shows expanded nodes in blue and the returned path in green.

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(grid, start, goal, heuristic=manhattan):
    """
    A* search on a GridWorld.

    Parameters
    ----------
    heuristic : callable(pos, goal) -> estimated cost

    Returns
    -------
    path    : list of (r, c) from start to goal, or None
    visited : set of expanded cells (cells popped from the heap)
    """
    # heap entries: (f_score, g_score, position, path_so_far)
    heap = [(heuristic(start, goal), 0, start, [start])]
    visited = {}   # pos -> best g seen

    while heap:
        f, g, pos, path = heapq.heappop(heap)
        if pos in visited:
            continue
        visited[pos] = g
        if pos == goal:
            return path, set(visited.keys())
        for nb in grid.neighbors(pos):
            if nb not in visited:
                ng = g + 1
                heapq.heappush(heap, (ng + heuristic(nb, goal), ng, nb, path + [nb]))

    return None, set(visited.keys())


astar_path, astar_visited = astar(gw, gw.start, gw.goal)
print(f"A*:  path length = {len(astar_path)}, cells expanded = {len(astar_visited)}")
print(f"BFS: path length = {len(bfs_path)},  cells expanded = {len(bfs_visited)}")
print(f"A* expanded {len(bfs_visited) - len(astar_visited)} fewer cells than BFS")
gw.render(path=astar_path, visited=astar_visited,
          title=f"A* (expanded={len(astar_visited)}, path={len(astar_path)})")

## Frontier Extraction and Selection

When the map is unknown the agent cannot plan to the goal. Instead it finds
**frontiers**: free known cells that border at least one unknown cell. Navigating
to a frontier expands the known map.

`extract_frontiers(known_map)` returns all frontier cells.
`select_frontier(frontiers, robot_pos)` picks the nearest one (by Manhattan distance).

In [ ]:
def extract_frontiers(known_map):
    """
    Extract frontier cells from a partially-known map.

    Parameters
    ----------
    known_map : 2-D numpy array
        -1 = unknown, 0 = free (known), 1 = wall (known)

    Returns
    -------
    List of (r, c) tuples that are free-known and adjacent to at least
    one unknown cell.
    """
    rows, cols = known_map.shape
    frontiers = []
    for r in range(rows):
        for c in range(cols):
            if known_map[r, c] != 0:          # must be known-free
                continue
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and known_map[nr, nc] == -1:
                    frontiers.append((r, c))
                    break
    return frontiers


def select_frontier(frontiers, robot_pos):
    """
    Select the frontier cell nearest to the robot (Manhattan distance).

    Returns None if the list is empty.
    """
    if not frontiers:
        return None
    return min(frontiers, key=lambda f: manhattan(f, robot_pos))


# ---- Quick demo on a partially-known map --------------------------------
demo_known = np.full((8, 8), -1)   # everything unknown
# Simulate the robot having revealed a small region
for r in range(4):
    for c in range(4):
        demo_known[r, c] = gw.grid[r, c]   # copy truth into known patch

frontiers = extract_frontiers(demo_known)
nearest = select_frontier(frontiers, robot_pos=(0, 0))
print(f"Frontiers found: {len(frontiers)}")
print(f"Nearest frontier to (0,0): {nearest}")

# Visualise
fig, ax = plt.subplots(figsize=(5, 5))
cmap_data = np.where(demo_known == -1, 0.5,
            np.where(demo_known == 1,  0.0, 1.0))
ax.imshow(cmap_data, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
fr, fc = zip(*frontiers) if frontiers else ([], [])
ax.scatter(fc, fr, c="cyan", s=80, label="Frontiers", zorder=3)
if nearest:
    ax.scatter(nearest[1], nearest[0], c="red", s=140, marker="*",
               label="Selected frontier", zorder=4)
ax.scatter(0, 0, c="orange", s=140, marker="^", label="Robot", zorder=4)
ax.legend(fontsize=8)
ax.set_title("Frontier extraction (gray=unknown, white=free, black=wall)")
ax.axis("off")
plt.tight_layout()
plt.show()

## Full Frontier-Based Exploration Loop

The agent starts with an empty map. At every step it:

1. Reveals nearby cells within `vision_radius` (simulating a sensor).
2. Extracts frontiers from the current known map.
3. Selects the nearest frontier.
4. Plans a path to it using A* over the known map.
5. Follows the path one step at a time, revealing more cells.

Coverage percentage (free cells known / total free cells) is recorded at each step.

In [ ]:
def run_frontier_exploration(world, vision_radius=2, max_steps=2000):
    """
    Full frontier-based exploration loop.

    Parameters
    ----------
    world         : GridWorld instance
    vision_radius : cells revealed around agent each step
    max_steps     : hard cap on total steps

    Returns
    -------
    coverage_history : list of float, coverage % after each step
    """
    rows, cols = world.height, world.width
    total_free = int(np.sum(world.grid == 0))

    # -1=unknown, 0=free, 1=wall
    known = np.full((rows, cols), -1, dtype=int)
    agent = world.start
    coverage_history = []

    def reveal(pos):
        r, c = pos
        for dr in range(-vision_radius, vision_radius + 1):
            for dc in range(-vision_radius, vision_radius + 1):
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols:
                    known[nr, nc] = world.grid[nr, nc]

    def astar_on_known(start, goal):
        """A* restricted to known-free cells."""
        heap = [(manhattan(start, goal), 0, start, [start])]
        vis = {}
        while heap:
            _, g, pos, path = heapq.heappop(heap)
            if pos in vis:
                continue
            vis[pos] = g
            if pos == goal:
                return path
            r, c = pos
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r + dr, c + dc
                if (0 <= nr < rows and 0 <= nc < cols
                        and known[nr, nc] == 0 and (nr, nc) not in vis):
                    heapq.heappush(heap,
                        (g + 1 + manhattan((nr,nc), goal), g+1,
                         (nr, nc), path + [(nr, nc)]))
        return None

    reveal(agent)
    steps = 0
    blocked_count = 0

    while steps < max_steps:
        known_free = int(np.sum(known == 0))
        coverage_history.append(known_free / total_free * 100)

        frontiers = extract_frontiers(known)
        if not frontiers:
            break

        target = select_frontier(frontiers, agent)
        path = astar_on_known(agent, target)

        if path is None or len(path) <= 1:
            # Mark this frontier unreachable and try the next nearest
            frontiers = [f for f in frontiers if f != target]
            blocked_count += 1
            if blocked_count > 10 or not frontiers:
                break
            continue

        blocked_count = 0
        for pos in path[1:]:
            agent = pos
            reveal(agent)
            steps += 1
            known_free = int(np.sum(known == 0))
            coverage_history.append(known_free / total_free * 100)
            if steps >= max_steps:
                break

    final_coverage = int(np.sum(known == 0)) / total_free * 100
    print(f"  Final coverage: {final_coverage:.1f}%  in {steps} steps")
    return coverage_history


# Use a 20x20 random world
def random_grid(rows, cols, wall_prob=0.20, seed=42):
    rng = np.random.default_rng(seed)
    grid = (rng.random((rows, cols)) < wall_prob).astype(int)
    grid[0, 0] = 0
    grid[rows-1, cols-1] = 0
    return grid


big_grid = random_grid(20, 20, wall_prob=0.18)
big_world = GridWorld(width=20, height=20,
                      obstacles=list(zip(*np.where(big_grid == 1))),
                      start=(0, 0), goal=(19, 19))

print("Frontier-based exploration (vision radius 2):")
cov_frontier = run_frontier_exploration(big_world, vision_radius=2)

## Coverage vs. Steps Plot

Plot how quickly the frontier strategy reaches full coverage.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(len(cov_frontier)), cov_frontier, label="Frontier-based (radius=2)")
ax.axhline(100, color="gray", linestyle="--", alpha=0.5, label="100% coverage")
ax.set_xlabel("Steps")
ax.set_ylabel("Coverage (%)")
ax.set_title("Frontier-based exploration: coverage over time")
ax.set_ylim(0, 105)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Comparison: Random Walk vs. Frontier-Based

A random walk also explores, but inefficiently. It wastes many steps revisiting
known areas. Below we run both strategies on the same world and plot their
coverage curves side by side.

In [ ]:
def run_random_walk(world, vision_radius=2, max_steps=2000, seed=0):
    """Coverage history for a random-walk explorer."""
    rng = np.random.default_rng(seed)
    rows, cols = world.height, world.width
    total_free = int(np.sum(world.grid == 0))
    known = np.full((rows, cols), -1, dtype=int)
    agent = world.start
    coverage_history = []

    def reveal(pos):
        r, c = pos
        for dr in range(-vision_radius, vision_radius + 1):
            for dc in range(-vision_radius, vision_radius + 1):
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols:
                    known[nr, nc] = world.grid[nr, nc]

    reveal(agent)
    for step in range(max_steps):
        coverage_history.append(int(np.sum(known == 0)) / total_free * 100)

        # Attempt a random move; retry up to 10 times to avoid walls
        r, c = agent
        for _ in range(10):
            dr, dc = world.ACTIONS[rng.integers(4)]
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and world.grid[nr, nc] == 0:
                agent = (nr, nc)
                break
        reveal(agent)

    coverage_history.append(int(np.sum(known == 0)) / total_free * 100)
    print(f"  Random walk final coverage: {coverage_history[-1]:.1f}%  in {max_steps} steps")
    return coverage_history


N_STEPS = 600
print("Random walk:")
cov_random = run_random_walk(big_world, vision_radius=2, max_steps=N_STEPS)
print("Frontier-based:")
cov_frontier2 = run_frontier_exploration(big_world, vision_radius=2, max_steps=N_STEPS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].plot(range(len(cov_random)), cov_random, color="tab:red")
axes[0].set_title("Random Walk")
axes[0].set_xlabel("Steps")
axes[0].set_ylabel("Coverage (%)")
axes[0].set_ylim(0, 105)
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(len(cov_frontier2)), cov_frontier2, color="tab:blue")
axes[1].set_title("Frontier-Based")
axes[1].set_xlabel("Steps")
axes[1].set_ylim(0, 105)
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.axhline(100, color="gray", linestyle="--", alpha=0.4)

plt.suptitle("Coverage curves: random walk vs. frontier-based (same world, vision radius=2)")
plt.tight_layout()
plt.show()

## Exercise: Greedy Frontier by Information Gain

The nearest-frontier heuristic ignores how much new area a frontier would reveal.
A smarter strategy scores each frontier by **information gain**: the number of
unknown cells within `vision_radius` of that frontier. This prefers frontiers
that will uncover large unexplored patches rather than thin peninsulas.

Implement `greedy_frontier(frontiers, robot_pos, known_map, vision_radius)`
that returns the frontier maximising information gain (breaking ties by distance).
Then replace `select_frontier` with your function and compare the coverage curve.

In [ ]:
def greedy_frontier(frontiers, robot_pos, known_map, vision_radius=2):
    """
    Select the frontier cell that maximises information gain.

    Information gain for a frontier cell f = number of cells within
    vision_radius of f that are currently unknown (known_map == -1).
    Ties are broken by Manhattan distance to robot_pos (prefer nearer).

    Parameters
    ----------
    frontiers     : list of (r, c) frontier cells
    robot_pos     : current agent position (r, c)
    known_map     : 2-D numpy array (-1=unknown, 0=free, 1=wall)
    vision_radius : sensor range

    Returns
    -------
    Best frontier cell (r, c), or None if list is empty.
    """
    if not frontiers:
        return None

    # YOUR CODE HERE
    raise NotImplementedError